### Setup

In [1]:
import matplotlib.pyplot as plt
import scipy as sc
from scipy.stats import pearsonr
import numpy as np
import pandas as pd
import seaborn as sns
import sys
import os
import torch
from tqdm import tqdm

In [2]:
sys.path.insert(0, os.getcwd())

from model import GPTConfig, GPT
from lib.utils import get_batch

In [3]:
device='cpu'
dataset='shakespeare_char'
checkpoint_dir='out-shakespeare-char'

### Identifiability

In [4]:
%%capture
torch.manual_seed(1337)

# load the checkpointed model state from last train save
ckpt_path = os.path.join(checkpoint_dir, 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

model.eval() # disables dropout
model.to(device)

In [5]:
model.config

GPTConfig(block_size=32, vocab_size=65, n_layer=2, n_head=3, head_size=20, batch_size=16, n_embd=64, dropout=0.2, bias=False, mlp_width=256)

**No LRB:**

GPTConfig(block_size=32, vocab_size=65, n_layer=2, n_head=3, head_size=64, batch_size=16, n_embd=64, dropout=0.2, bias=False, mlp_width=256)

**LRB:**



In [6]:
n = model.config.block_size
h = model.config.n_head
hs = model.config.head_size

In [7]:
X, Y = get_batch('eval', os.path.join('data', dataset), device, n, model.config.batch_size)
A,T,v = model.get_matricies(X,0)

In [8]:
T.shape

(32, 20)

In [9]:
np.linalg.matrix_rank(T)

np.int64(20)

In [10]:
T_aug = np.concatenate([T, np.ones((n, 1))], axis=1)
basis_LN = sc.linalg.null_space(T_aug.T)

In [11]:
basis_LN.shape

(32, 11)

In [12]:
d = basis_LN[:,0]
a = A[:,1]

In [13]:
a.shape

(32,)

In [14]:
def get_lam_constraints(d):
    lam_max_list = []
    lam_min_list = []
    for idx in range(len(d)):
        if d[idx] < 0:
            lam_max_list.append(-a[idx] / d[idx])
        elif d[idx] > 0:
            lam_min_list.append(-a[idx] / d[idx])
    
    lam_max = np.min(lam_max_list)
    lam_min = np.max(lam_min_list)
    return lam_min, lam_max

In [24]:
for i in range(basis_LN.shape[1]):
    d = basis_LN[:,i]
    lam_min, lam_max = get_lam_constraints(d)
    
    for lam in np.linspace(lam_min, lam_max, 1000):
        a_prime = a + lam * d
        A[:,0] = a_prime
        
        vals = np.zeros(basis_LN.shape[1])
        for j in range(basis_LN.shape[1]):
            vals[j]= basis_LN[:,j].T  @ A[:,0] / basis_LN[:,j].T @ np.ones(n)

        print(np.linalg.norm(np.diff(vals)))

27.960108643237945
27.967220732238854
27.974788703600133
27.982825077044314
27.99132459095628
28.000271306849857
28.0096925140172
28.0195774705713
28.029920939565827
28.040715842340212
28.051968142960465
28.063695171430947
28.075861748916452
28.08849223326846
28.101584356607578
28.1151228235621
28.12912905027008
28.143587752135318
28.158479806196254
28.173847046382555
28.18966804258605
28.205937459354793
28.222648508171947
28.239818260925382
28.257431937597378
28.275490781862985
28.294004124440786
28.312964062455617
28.332369832599458
28.35223074336532
28.372514109659512
28.393262932545234
28.41443011204866
28.43605197612004
28.458116156577784
28.480609046821762
28.503548232048377
28.52693577638306
28.550740258109897
28.574984419632987
28.59966260895549
28.62477753550155
28.650320211554785
28.676292578459897
28.70270332155972
28.72953198212625
28.756792252108777
28.7844723658448
28.812585182312187
28.841114956923736
28.8700710239883
28.899440751085976
28.92923347465626
28.9594451480434

np.float64(-1.6870864878366236)

In [16]:
basis_LN[:,0].T  @ A[:,0] / basis_LN[:,0].T @ np.ones(n)

np.float64(-68.41083808214918)

In [ ]:
check_val = basis_LN[:,0].T  @ A[:,0] / basis_LN[:,0].T @ np.ones(n)
vals_prime = np.zeros(basis_LN.shape[1])

for j in range(basis_LN.shape[1]):
    vals_prime[j]= basis_LN[:,j].T  @ A[:,20] / basis_LN[:,j].T @ np.ones(n)
vals_prime

In [ ]:
np.linalg.norm(np.diff(vals_prime))

In [ ]:
np.linalg.norm(np.diff(vals))